# 01 Build Dataset

Build the dataset from `source_manifest.csv`.

This notebook:
- Loads the manifest
- Converts active manifest rows into document rows
- Attaches local raw text when available
- Creates a packet-level link scaffold
- Writes `documents.csv` and `links.csv`


In [1]:
from pathlib import Path
import sys

repo_root = Path('..').resolve()
sys.path.insert(0, str(repo_root / 'src'))

from normalize import DOCUMENT_FIELDS, load_manifest, active_manifest_rows, manifest_to_documents, write_csv
from extract import attach_raw_text
from link_documents import LINK_FIELDS, build_link_scaffold

manifest_path = repo_root / 'manifests' / 'source_manifest.csv'
raw_root = repo_root / 'data' / 'raw'
processed_root = repo_root / 'data' / 'processed'


In [2]:
manifest_rows = load_manifest(manifest_path)
active_rows = active_manifest_rows(manifest_rows)

print({'manifest_rows': len(manifest_rows), 'active_rows': len(active_rows)})


{'manifest_rows': 32, 'active_rows': 29}


In [3]:
documents = manifest_to_documents(active_rows)
documents = attach_raw_text(documents, raw_root)
links = build_link_scaffold(documents)

write_csv(processed_root / 'documents.csv', DOCUMENT_FIELDS, documents)
write_csv(processed_root / 'links.csv', LINK_FIELDS, links)

print({'documents_written': len(documents), 'links_written': len(links)})


{'documents_written': 29, 'links_written': 24}


In [4]:
import pandas as pd

documents_df = pd.DataFrame(documents)
links_df = pd.DataFrame(links)

documents_df['has_text'] = documents_df['text_clean'].fillna('').str.len() > 0
display(documents_df.groupby(['packet_id', 'layer', 'has_text']).size().reset_index(name='count'))
display(links_df.head(10))


,packet_id,layer,has_text,count
0,CN-02,news,True,3
1,CN-02,official,True,1
2,CN-02,social,True,1
3,CN-04,news,True,3
4,CN-04,official,True,1
5,CN-04,social,True,3
6,US-01,news,True,3
7,US-01,official,True,1
8,US-02,news,True,3
9,US-02,official,True,1


,link_id,from_doc_id,to_doc_id,relation_type,evidence,confidence,notes
0,us-01__news__1,us-01__news__news_1,us-01__official__anchor,reports_on,official_news_release_on_same_guidance,0.95,Official NewsNet release tied directly to the ...
1,us-01__news__2,us-01__news__news_2,us-01__official__anchor,reports_on,workflow_explainer_on_same_guidance,0.90,Official explainer for the same guidance and r...
2,us-01__news__3,us-01__news__news_3,us-01__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
3,us-02__news__1,us-02__news__news_1,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
4,us-02__news__2,us-02__news__news_2,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
5,us-02__news__3,us-02__news__news_3,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
6,us-02__social__1,us-02__social__social_1,us-02__official__anchor,reposts_or_discusses,packet_membership,0.20,Packet-level link; individual verification pen...
7,us-02__social__2,us-02__social__social_2,us-02__official__anchor,reposts_or_discusses,packet_membership,0.20,Packet-level link; individual verification pen...
8,us-03__news__1,us-03__news__news_1,us-03__official__anchor,reports_on,official_news_release_on_same_report,0.98,Official NewsNet release announcing the same P...
9,us-03__news__2,us-03__news__news_2,us-03__official__anchor,reports_on,general_news_on_same_report,0.85,General news coverage summarizing the same Cop...
